In [29]:
import numpy as np
import pandas as pd
from statsmodels.formula.api import logit
from statsmodels.formula.api import ols
from sklearn.metrics import recall_score
from scipy import stats

### 문제1

In [2]:
df = pd.read_csv('elderly_health.csv')
df.head()

,id,age,diabetic,activity,glus_fast,bmi,blood_pressure,predicted
0,1,70,no,no,154,26.8,136,1
1,2,58,no,yes,113,23.7,136,0
2,3,55,no,no,156,25.5,141,0
3,4,50,no,yes,183,29.8,100,0
4,5,40,no,no,122,29.9,125,1


In [3]:
train = df[:2000].copy()
test = df[2000:].copy()

In [8]:
model = logit('predicted ~ age + diabetic + activity + glus_fast + bmi + blood_pressure', data=train).fit()
model.summary()

Optimization terminated successfully.
         Current function value: 0.486192
         Iterations 7


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:              predicted   No. Observations:                 2000
Model:                          Logit   Df Residuals:                     1992
Method:                           MLE   Df Model:                            7
Date:                Wed, 18 Nov 2026   Pseudo R-squ.:                  0.2883
Time:                        14:52:08   Log-Likelihood:                -972.38
converged:                       True   LL-Null:                       -1366.3
Covariance Type:            nonrobust   LLR p-value:                7.486e-166
===================================================================================
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -6.7577      0.754     -8.960      0.000      -8.236      -5.279
diabetic[T.pre]     1.3780      0.133     10.376      0.000       1.118       1.638
diabetic[T.yes]     3.0138      0.231     13.055      0.000       2.561       3.466
activity[T.yes]    -1.4809      0.121    -12.246      0.000      -1.718      -1.244
age                 0.0869      0.005     15.991      0.000       0.076       0.098
glus_fast           0.0211      0.002     11.304      0.000       0.017       0.025
bmi                -0.0171      0.015     -1.160      0.246      -0.046       0.012
blood_pressure     -0.0017      0.004     -0.450      0.653      -0.009       0.006
===================================================================================
"""

In [10]:
odds = np.exp(model.params['diabetic[T.yes]'])
print("diabetic yes 오즈비 :", round(odds,2))

diabetic yes 오즈비 : 20.36


In [11]:
pred = model.predict(test)
idx = pred.idxmax()
print("노령층일 확률이 가장 큰 사람의 공복 혈당 :", test.loc[idx,'glus_fast'])

노령층일 확률이 가장 큰 사람의 공복 혈당 : 181


In [13]:
pred =(pred > 0.2).astype(int)

recall = recall_score(test['predicted'], pred)
print("recall : ", round(recall,2))

recall :  0.94


### 문제2

In [16]:
df = pd.read_csv('promotion_data.csv')
df.head()

,sales,visit_count,page_time,ad_clicks,promotion_budget
0,35341.611727,36,168,18,11340
1,64757.849132,32,214,21,11224
2,40615.684241,56,172,6,5468
3,52009.480757,70,234,5,7486
4,32462.259065,27,88,8,7547


In [18]:
statistic, pvalue = stats.ttest_1samp(df['sales'], 35000)
print("검정통계량 :", statistic)
print("pvalue :", round(pvalue,3))

검정통계량 : 1.3826844415882558
pvalue : 0.168


In [22]:
corr_df = df.corr()
corr_df

,sales,visit_count,page_time,ad_clicks,promotion_budget
sales,1.000000,0.044154,0.575396,-0.131915,-0.091270
visit_count,0.044154,1.000000,0.011349,0.078189,-0.089217
page_time,0.575396,0.011349,1.000000,-0.093688,-0.039554
ad_clicks,-0.131915,0.078189,-0.093688,1.000000,0.036940
promotion_budget,-0.091270,-0.089217,-0.039554,0.036940,1.000000


In [28]:
print("상관계수가 가장 큰 컬럼 :", corr_df['sales'][1:].idxmax())
print("상관계수가 가장 큰 값 :", round(corr_df['sales'][1:].max(),3))

상관계수가 가장 큰 컬럼 : page_time
상관계수가 가장 큰 값 : 0.575


In [32]:
model = ols('sales ~ visit_count + page_time + ad_clicks + promotion_budget',data=df).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  sales   R-squared:                       0.343
Model:                            OLS   Adj. R-squared:                  0.330
Method:                 Least Squares   F-statistic:                     25.45
Date:                Wed, 18 Nov 2026   Prob (F-statistic):           5.60e-17
Time:                        15:20:20   Log-Likelihood:                -2160.2
No. Observations:                 200   AIC:                             4330.
Df Residuals:                     195   BIC:                             4347.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
====================================================================================
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept         2.234e+04   4136.622      5.400      0.000    1.42e+04    3.05e+04
visit_count         28.7893     43.854      0.656      0.512     -57.701     115.279
page_time          130.7392     13.501      9.684      0.000     104.113     157.365
ad_clicks         -116.0846     85.288     -1.361      0.175    -284.290      52.121
promotion_budget    -0.3601      0.336     -1.072      0.285      -1.023       0.303
==============================================================================
Omnibus:                        0.318   Durbin-Watson:                   1.941
Prob(Omnibus):                  0.853   Jarque-Bera (JB):                0.286
Skew:                          -0.091   Prob(JB):                        0.867
Kurtosis:                       2.970   Cond. No.                     3.63e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 3.63e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [37]:
pvalue = model.pvalues.sort_values(ascending=True)
pvalue

page_time           2.339716e-18
Intercept           1.922269e-07
ad_clicks           1.750570e-01
promotion_budget    2.851844e-01
visit_count         5.122933e-01
dtype: float64

In [39]:
idx = pvalue.index[0]
print("회귀계수 :", round(model.params[idx],3))

회귀계수 : 130.739
